In [1]:
%%capture
!pip install tensorboard
!pip install nltk
!pip install evaluate
!pip install transformers
!pip install datasets
!pip install sacrebleu accelerate
!pip install git+https://github.com/csebuetnlp/normalizer


In [2]:
%%capture
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from normalizer import normalize
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
import evaluate
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from transformers.integrations import TensorBoardCallback
from transformers import TrainerCallback
# Ensure the tokenizer data is downloaded (required in Kaggle/Colab)
nltk.download('punkt')
nltk.download('punkt_tab') # Required for newer NLTK versions
# Verify installation
import transformers
print(f"Transformers version: {transformers.__version__}")

2026-01-16 09:51:42.574798: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768557102.798689      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768557102.863664      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768557103.420669      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768557103.420707      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768557103.420710      55 computation_placer.cc:177] computation placer alr

In [3]:
tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M", src_lang="ben_Beng" , tgt_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [4]:
for name, param in model.model.encoder.named_parameters():
    if param.requires_grad:
        print(f"{name} is trainable")
    else:
        print(f"{name} is frozen")

embed_tokens.weight is trainable
layers.0.self_attn.k_proj.weight is trainable
layers.0.self_attn.k_proj.bias is trainable
layers.0.self_attn.v_proj.weight is trainable
layers.0.self_attn.v_proj.bias is trainable
layers.0.self_attn.q_proj.weight is trainable
layers.0.self_attn.q_proj.bias is trainable
layers.0.self_attn.out_proj.weight is trainable
layers.0.self_attn.out_proj.bias is trainable
layers.0.self_attn_layer_norm.weight is trainable
layers.0.self_attn_layer_norm.bias is trainable
layers.0.fc1.weight is trainable
layers.0.fc1.bias is trainable
layers.0.fc2.weight is trainable
layers.0.fc2.bias is trainable
layers.0.final_layer_norm.weight is trainable
layers.0.final_layer_norm.bias is trainable
layers.1.self_attn.k_proj.weight is trainable
layers.1.self_attn.k_proj.bias is trainable
layers.1.self_attn.v_proj.weight is trainable
layers.1.self_attn.v_proj.bias is trainable
layers.1.self_attn.q_proj.weight is trainable
layers.1.self_attn.q_proj.bias is trainable
layers.1.self_att

In [5]:
# Freeze all encoder layers initially
for param in model.model.encoder.parameters():
    param.requires_grad = False

print("All encoder layers frozen initially.")

All encoder layers frozen initially.


In [6]:
from transformers import TrainerCallback

class GradualUnfreezingCallback(TrainerCallback):
    """
    Gradually unfreeze encoder layers during training.
    unfreeze_schedule: dict mapping epoch -> number of encoder layers to unfreeze
    Example: {1: 2, 2: 4, 3: 'all'}
    """
    def __init__(self, model, unfreeze_schedule):
        self.model = model
        self.unfreeze_schedule = unfreeze_schedule
        self.encoder_layers = model.model.encoder.layers
        self.total_layers = len(self.encoder_layers)

    def on_epoch_begin(self, args, state, control, **kwargs):
        epoch = int(state.epoch)
        if epoch not in self.unfreeze_schedule:
            return

        layers_to_unfreeze = self.unfreeze_schedule[epoch]

        if layers_to_unfreeze == "all":
            start = 0
        else:
            # Unfreeze the top `layers_to_unfreeze` layers
            start = self.total_layers - layers_to_unfreeze

        print(f"\n🔓 Unfreezing encoder layers {start} to {self.total_layers - 1} at epoch {epoch}")

        for layer in self.encoder_layers[start:]:
            for param in layer.parameters():
                param.requires_grad = True


In [7]:
tensorboard_callback = TensorBoardCallback()

In [8]:
max_length = 64

In [10]:
def tokenize_and_create_dataset(tokenizer, data_df, max_length=None):
    # Tokenize English text
    normalized_bangla = [normalize(text) for text in data_df["bangla_speech"]]
    encodings = tokenizer(
        # list(data_df["bangla_speech"],
        normalized_bangla,    #may remove the normalization here
        truncation=True,
        padding=True,
        max_length=max_length  
    )

    # Tokenize Egyptian text
    decodings = tokenizer(
        list(data_df["english_speech"]),
        truncation=True,
        padding=True,
        max_length=max_length  
    )

    # Create Dataset object
    dataset = Dataset.from_dict({
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "labels": decodings["input_ids"],
    })

    return dataset

In [11]:
# Path to your CSV file
csv_path = "/kaggle/input/train-fin-vashantor/all_train_dataset.csv"

# Load CSV into a DataFrame
train_df = pd.read_csv(csv_path)

# Show the first few rows to verify
print(train_df.head())

# Check the total number of rows and columns
print(f"DataFrame shape: {train_df.shape}")

                          bangla_speech                        english_speech  \
0                      লকট মোর প্রিয় ফল           Jamrul is my favorite fruit   
1  যত হারাহারি সম্ভব বিয়ার ব্যবস্থা গরো  Arrange marriage as soon as possible   
2   যত তাড়াতাড়ি সম্ভব বিয়ার ব্যবস্থা কর  Arrange marriage as soon as possible   
3  যত তাড়াতাড়ি সম্ভব বিয়ার ব্যবস্থা করো  Arrange marriage as soon as possible   
4      যত জলদি সম্ভব বিয়ার ব্যবস্থা খরো  Arrange marriage as soon as possible   

      dialect  
0   Barishal   
1  Chittagong  
2  Mymensingh  
3    Noakhali  
4      Sylhet  
DataFrame shape: (9374, 3)


In [13]:
import pandas as pd

def load_translation_dataset(csv_path, bangla_col):
    """
    Load a translation CSV and return a standardized DataFrame.

    Args:
        csv_path (str): Path to the CSV file.
        bangla_col (str): Name of the Bangla sentence column in the CSV.
        english_col (str): Name of the English sentence column in the CSV.

    Returns:
        pd.DataFrame: DataFrame with columns ['bangla_speech', 'english_speech'].
    """
    english_col = "english_speech"
    # Load CSV
    df = pd.read_csv(csv_path)

    # Strip whitespace from column names
    df.columns = df.columns.str.strip()

    # Keep only the relevant columns
    df = df[[bangla_col.strip(), english_col.strip()]]

    # Rename columns to standard names
    df = df.rename(columns={bangla_col.strip(): "bangla_speech",
                            english_col.strip(): "english_speech"})

    return df



In [14]:
# Paths to the validation CSVs
barishal_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Barishal  Validation Translation.csv"
chittagong_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Chittagong Validation Translation.csv"
mymensingh_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Mymensingh Validation Translation.csv"
noakhali_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Noakhali Validation Translation.csv"
sylhet_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Validation/Sylhet Validation Translation.csv"

# Load datasets using your function
barishal_valid_df = load_translation_dataset(barishal_csv, "barishal_bangla_speech ")
chittagong_valid_df = load_translation_dataset(chittagong_csv, "chittagong_bangla_speech ")
mymensingh_valid_df = load_translation_dataset(mymensingh_csv, "mymensingh_bangla_speech ")
noakhali_valid_df = load_translation_dataset(noakhali_csv, "noakhali_bangla_speech ")
sylhet_valid_df = load_translation_dataset(sylhet_csv, "sylhet_bangla_speech ")
standard_valid_df = load_translation_dataset(sylhet_csv, "bangla_speech")

In [15]:
barishal_test_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Test/Barishal Test Translation.csv"
chittagong_test_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Test/Chittagong Test Translation.csv"
mymensingh_test_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Test/Mymensingh Test Translation.csv"
noakhali_test_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Test/Noakhali Test Translation.csv"
sylhet_test_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Test/Sylhet Test Translation.csv"
standard_test_csv = "/kaggle/input/vashantor-dialect-to-english/Vashantor /Test/Sylhet Test Translation.csv"

# Load test datasets
barishal_test_df = load_translation_dataset(barishal_test_csv, "barishal_bangla_speech ")
chittagong_test_df = load_translation_dataset(chittagong_test_csv, "chittagong_bangla_speech ")
mymensingh_test_df = load_translation_dataset(mymensingh_test_csv, "mymensingh_bangla_speech ")
noakhali_test_df = load_translation_dataset(noakhali_test_csv, "noakhali_bangla_speech ")
sylhet_test_df = load_translation_dataset(sylhet_test_csv, "sylhet_bangla_speech ")
standard_test_df = load_translation_dataset(standard_test_csv, "bangla_speech")

In [17]:
train_dataset = tokenize_and_create_dataset(tokenizer, train_df)

In [19]:
barishal_valid_dataset = tokenize_and_create_dataset(tokenizer,barishal_valid_df)
chittagong_valid_dataset = tokenize_and_create_dataset(tokenizer, chittagong_valid_df)
mymensingh_valid_dataset = tokenize_and_create_dataset(tokenizer, mymensingh_valid_df)
noakhali_valid_dataset = tokenize_and_create_dataset(tokenizer, noakhali_valid_df)
sylhet_valid_dataset = tokenize_and_create_dataset(tokenizer, sylhet_valid_df)
standard_valid_dataset = tokenize_and_create_dataset(tokenizer, standard_valid_df)

In [ ]:
barishal_test_dataset = tokenize_and_create_dataset(barishal_test_df)
chittagong_test_dataset = tokenize_and_create_dataset(chittagong_test_df)
mymensingh_test_dataset = tokenize_and_create_dataset(mymensingh_test_df)
noakhali_test_dataset = tokenize_and_create_dataset(noakhali_test_df)
sylhet_test_dataset = tokenize_and_create_dataset(sylhet_test_df)
standard_test_dataset = tokenize_and_create_dataset(standard_test_df)

In [29]:
import shutil
import os

# Delete the cached evaluate metrics folder
cache_dir = os.path.expanduser("~/.cache/huggingface/modules/evaluate_modules")
shutil.rmtree(cache_dir, ignore_errors=True)
print("✅ Hugging Face evaluate cache cleared.")

✅ Hugging Face evaluate cache cleared.


In [40]:
!pip install sacrebleu

import sacrebleu

def decode_strip_pad(tokenizer, sequences):
    """
    Decode token IDs into strings after removing padding tokens.
    sequences: list of list of token IDs
    """
    decoded_texts = []
    for seq in sequences:
        # Remove padding tokens before decoding
        seq = [tok for tok in seq if tok != tokenizer.pad_token_id]
        decoded_texts.append(tokenizer.decode(seq, skip_special_tokens=True))
    return decoded_texts

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    # Decode predictions and labels, stripping pad tokens
    decoded_preds = decode_strip_pad(tokenizer, preds)
    decoded_labels = decode_strip_pad(tokenizer, labels)

    # sacrebleu expects a list of references per prediction
    references = [[label] for label in decoded_labels]

    bleu_score = sacrebleu.corpus_bleu(decoded_preds, references).score
    return {"bleu": bleu_score}

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
# def compute_bleu(preds, labels, tokenizer):
#     """
#     preds: numpy array of predicted token IDs
#     labels: numpy array of label token IDs
#     tokenizer: your Hugging Face tokenizer
#     """
#     # decode predictions
#     decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    
#     # decode labels
#     decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
#     # sacrebleu expects a list of references for each sentence
#     decoded_labels = [[label] for label in decoded_labels]
    
#     result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
#     return result["score"]


In [33]:
# def decode_strip_pad(tokenizer, sequences):
#     """
#     Decode token IDs into strings after removing padding tokens.
#     sequences: list of list of token IDs
#     """
#     decoded_texts = []
#     for seq in sequences:
#         # Remove padding tokens before decoding
#         seq = [tok for tok in seq if tok != tokenizer.pad_token_id]
#         decoded_texts.append(tokenizer.decode(seq, skip_special_tokens=True))
#     return decoded_texts

In [ ]:
# def compute_metrics(eval_pred):
#     preds, labels = eval_pred
#     decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
#     decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
#     decoded_labels = [[label] for label in decoded_labels]  # sacrebleu format
#     result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
# #     return {"bleu": result["score"]}

# def compute_metrics(eval_pred):
#     preds, labels = eval_pred

#     # Remove padding before decoding
#     decoded_preds = decode_strip_pad(tokenizer, preds)
#     decoded_labels = decode_strip_pad(tokenizer, labels)

#     # sacrebleu expects a list of references for each prediction
#     decoded_labels = [[label] for label in decoded_labels]

#     result = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
#     return {"bleu": result["score"]}

In [41]:
val_datasets = {
    "Barishal": barishal_valid_dataset,
    "Chittagong": chittagong_valid_dataset,
    "Mymensingh": mymensingh_valid_dataset,
    "Noakhali": noakhali_valid_dataset,
    "Sylhet": sylhet_valid_dataset,
    "Standard": standard_valid_dataset
}

from datasets import concatenate_datasets

eval_dataset = concatenate_datasets([
    barishal_valid_dataset,
    chittagong_valid_dataset,
    mymensingh_valid_dataset,
    noakhali_valid_dataset,
    sylhet_valid_dataset,
    standard_valid_dataset
])

In [48]:
# from transformers import TrainerCallback, TrainerState, TrainerControl

# class MultiValidationBLEUCallback(TrainerCallback):
#     """
#     Evaluates the model on multiple validation datasets using BLEU at the end of each epoch.
#     """
#     def __init__(self, val_datasets, trainer, tokenizer):
#         self.val_datasets = val_datasets
#         self.trainer = trainer
#         self.tokenizer = tokenizer

#     def on_epoch_end(self, args, state: TrainerState, control: TrainerControl, **kwargs):
#         print(f"\n=== BLEU Evaluation at epoch {state.epoch:.2f} ===")
#         for name, dataset in self.val_datasets.items():
#             # make predictions
#             preds = self.trainer.predict(dataset).predictions
#             labels = self.trainer.predict(dataset).label_ids
#             bleu_score = compute_bleu(preds, labels, self.tokenizer)
#             print(f"{name} BLEU: {bleu_score:.2f}")
#         return control

# multi_val_bleu_cb = MultiValidationBLEUCallback(val_datasets, trainer=None, tokenizer=tokenizer)

from transformers import TrainerCallback, TrainerState, TrainerControl

class MultiValidationBLEUCallback(TrainerCallback):
    """
    Evaluates the model on multiple validation datasets using BLEU at the end of each epoch.
    Uses the provided compute_metrics function.
    """
    def __init__(self, val_datasets, trainer, compute_metrics_fn):
        self.val_datasets = val_datasets
        self.trainer = trainer
        self.compute_metrics_fn = compute_metrics_fn

    def on_epoch_end(self, args, state: TrainerState, control: TrainerControl, **kwargs):
        print(f"\n=== BLEU Evaluation at epoch {state.epoch:.2f} ===")
        for name, dataset in self.val_datasets.items():
            # Make predictions using the trainer
            outputs = self.trainer.predict(dataset)
            eval_pred = (outputs.predictions, outputs.label_ids)
            
            # Compute BLEU using the given compute_metrics function
            result = self.compute_metrics_fn(eval_pred)
            print(f"{name} BLEU: {result['bleu']:.2f}")
        return control

# Usage
multi_val_bleu_cb = MultiValidationBLEUCallback(
    val_datasets=val_datasets,
    trainer=trainer,
    compute_metrics_fn=compute_metrics  # your function from before
)

In [49]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [50]:
unfreeze_schedule = {
    1: 2,     # unfreeze top 2 layers at epoch 1
    2: 4,     # unfreeze top 4 layers at epoch 2
    3: 'all'  # unfreeze all layers at epoch 3
}

gradual_unfreeze_cb = GradualUnfreezingCallback(model=model, unfreeze_schedule=unfreeze_schedule)


In [51]:
model_args = Seq2SeqTrainingArguments(
    output_dir="./output_dir",
    per_device_train_batch_size=8,       # Increased from 1
    gradient_accumulation_steps=4,       # Effective batch size = 64 (8x4x2)
    per_device_eval_batch_size=8,
    learning_rate=5e-5,                  # Slightly higher for faster convergence
    warmup_ratio=0.1,
    num_train_epochs=4,                  
    weight_decay=0.01,
    lr_scheduler_type="linear",
    eval_strategy="steps",         # Eval once per epoch instead of every 10k steps
    eval_steps=1000,
    save_steps=1000,
    logging_steps=200,
    save_strategy="epoch",
    metric_for_best_model="bleu",   # 🔴 REQUIRED
    greater_is_better=True,  
    predict_with_generate=True,
    fp16=True,                           # CRITICAL: Use Mixed Precision (if using NVIDIA GPU)
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    tokenizer=tokenizer,
    args=model_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics, 
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=1,
            early_stopping_threshold=0.0
        ),
        gradual_unfreeze_cb ,  
        multi_val_bleu_cb
        
    ],
)

multi_val_bleu_cb.trainer = trainer
trainer.train()

/tmp/ipykernel_55/256542983.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss,Validation Loss



=== BLEU Evaluation at epoch 1.00 ===
Barishal BLEU: 15.11
Chittagong BLEU: 22.59
Mymensingh BLEU: 37.99
Noakhali BLEU: 37.99
Sylhet BLEU: 37.99
Standard BLEU: 37.99


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(



🔓 Unfreezing encoder layers 10 to 11 at epoch 1


...

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(



=== BLEU Evaluation at epoch 3.00 ===
Barishal BLEU: 37.99
Chittagong BLEU: 22.59
Mymensingh BLEU: 37.99
Noakhali BLEU: 37.99
Sylhet BLEU: 37.99
Standard BLEU: 37.99

🔓 Unfreezing encoder layers 0 to 11 at epoch 3


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(



=== BLEU Evaluation at epoch 4.00 ===
Barishal BLEU: 37.99
Chittagong BLEU: 22.59
Mymensingh BLEU: 37.99
Noakhali BLEU: 37.99
Sylhet BLEU: 37.99
Standard BLEU: 37.99


TrainOutput(global_step=588, training_loss=5.559310394079507, metrics={'train_runtime': 2288.4458, 'train_samples_per_second': 16.385, 'train_steps_per_second': 0.257, 'total_flos': 3808957226287104.0, 'train_loss': 5.559310394079507, 'epoch': 4.0})

In [1]:
# Suppose your trainer is called `trainer`
trainer.save_model("./finetuned_nllb600m")
tokenizer.save_pretrained("./finetuned_nllb600m")


NameError: name 'trainer' is not defined

In [ ]:
torch.cuda.empty_cache()

In [ ]:
import numpy as np

# 1. Preprocessing with -100 for padding labels
def preprocess_labels(labels_ids):
    # Replace tokenizer padding token id with -100
    return [[(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels_ids]

In [ ]:
model_args = Seq2SeqTrainingArguments(
    output_dir="./eval_results",
    per_device_eval_batch_size=16,      # Much faster than 1
    predict_with_generate=True,        # Necessary for real translation eval
    fp16=True,                         # Use if you have an NVIDIA GPU
    report_to="none"
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model, 
    args=model_args, 
    tokenizer=tokenizer
)